# Alignment Pipeline — Fixed

Fixes: runtime GPU, numpy conflict, audioop, taaldetectie, Drive copy, device check, checkpoint upgrade.

## 0. Runtime check

⚠️ Zorg dat je runtime op **GPU** staat: `Runtime → Change runtime type → T4 GPU`

(Notebook had `accelerator: TPU` — dat werkt niet met CUDA)

In [1]:
import torch
assert torch.cuda.is_available(), '❌ Geen GPU gevonden! Zet runtime op GPU via Runtime → Change runtime type'
print('✅ GPU:', torch.cuda.get_device_name(0))

✅ GPU: Tesla T4


## 1. Clone repo

In [2]:
import os
if not os.path.exists('/content/Video_Analyzer'):
    !git clone https://github.com/Yi-Star32/Video_Analyzer.git /content/Video_Analyzer
%cd /content/Video_Analyzer

/content/Video_Analyzer


## 2. Dependencies

Fix volgorde: whisperx eerst (trekt numpy 2.x), daarna niets dat downgradet.
`audioop-lts` werkt niet op Python 3.12 → gefilterd.

In [3]:
# Installeer requirements zonder audioop-lts (niet beschikbaar op Python 3.12)
!grep -v 'audioop-lts' requirements.txt > /tmp/requirements_fixed.txt
!pip install -q -r /tmp/requirements_fixed.txt

grep: requirements.txt: binary file matches


In [4]:
# Installeer whisperx (trekt numpy>=2.1 mee)
!pip install -q git+https://github.com/m-bain/whisperx.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 598.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.9 MB/s 

In [5]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.2 MB/s eta 0:00:00


In [3]:
# Verifieer numpy versie — moet >=2.1.0 zijn voor whisperx
import numpy as np
print('numpy:', np.__version__)
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 1), \
    f'❌ numpy {np.__version__} te oud voor whisperx — herstart runtime en run cellen opnieuw'
print('✅ numpy OK')

numpy: 2.4.6
✅ numpy OK


## 3. Upgrade Lightning checkpoint (eenmalig, elimineert herhaalde warning)

In [4]:
!python -m lightning.pytorch.utilities.upgrade_checkpoint \
    /usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin 2>/dev/null || true
print('✅ Checkpoint upgrade klaar (of al up-to-date)')

✅ Checkpoint upgrade klaar (of al up-to-date)


## 4. Imports

In [8]:
!pip uninstall -y transformers
!pip install --no-cache-dir transformers

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 197.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 437.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
whisperx 3.8.6 requires huggingface-hub<1.0.0, but you have huggingface-hub 1.18.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


In [5]:
import sys
import warnings
from pathlib import Path
from tqdm import TqdmWarning

warnings.filterwarnings('ignore', category=TqdmWarning)
warnings.filterwarnings('ignore', message='.*gradient_checkpointing.*')

project_path = '/content/Video_Analyzer'
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio
print('✅ Imports OK')

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✅ Imports OK


## 5. Google Drive koppelen + audio naar lokale SSD kopiëren

Drive-reads zijn traag (~50 MB/s). Kopieer audio eenmalig naar `/content/audio/` voor 3-5x snelere verwerking.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive/projecten/Video_Analyzer_data')
LOCAL_AUDIO  = Path('/content/audio')
SONG_PATH    = DRIVE_ROOT / 'output/separated/htdemucs/audio/vocals.wav'

assert DRIVE_ROOT.exists(), f'Drive root niet gevonden: {DRIVE_ROOT}'
assert SONG_PATH.exists(),  f'vocals.wav niet gevonden: {SONG_PATH}'
print('✅ Drive OK, song gevonden')

Mounted at /content/drive
✅ Drive OK, song gevonden


In [7]:
# Kopieer episodes naar lokale SSD (alleen als nog niet gedaan)
import shutil

DRIVE_EPISODES = DRIVE_ROOT / 'data/episodes_audio'
LOCAL_AUDIO.mkdir(exist_ok=True)

copied = 0
for src in DRIVE_EPISODES.rglob('audio.wav'):
    dst = LOCAL_AUDIO / src.parent.name / 'audio.wav'
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1

files = sorted(LOCAL_AUDIO.rglob('audio.wav'))
print(f'✅ {len(files)} bestanden beschikbaar lokaal ({copied} nieuw gekopieerd)')

✅ 30 bestanden beschikbaar lokaal (30 nieuw gekopieerd)


## 6. Models initialiseren

Fixes:
- `language='en'` → geen detectie per bestand (~60s bespaard per episode)
- `AudioEmbeddingPipeline(device=device)` → Wav2Vec2 op GPU
- `compute_type='float16'` → sneller op moderne GPU

In [8]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')

# FIX: geef device mee aan pipeline zodat Wav2Vec2 ook op GPU draait
pipeline = AudioEmbeddingPipeline(device=device)

# FIX: language='en' voorkomt detectie per bestand
# FIX: compute_type='float16' voor ~2x snelheid op GPU
aligner = PhonemeAligner(
    device=device,
    whisper_model='base',
    language='en',
    compute_type='float16',
)
print('✅ Models geladen')

Device: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

✅ Models geladen


## 7. Phoneme index bouwen

In [ ]:
# Batch embed_audio patch — verwerkt 64 phonemes tegelijk ipv 1 per keer
import numpy as np
import torch
from scipy.signal import resample as scipy_resample
import audio_matcher.alignment as _align_mod

def _fixed_load_audio(file_path: str, sr: int = 16000):
    import soundfile as _sf
    audio, orig_sr = _sf.read(file_path, always_2d=True)
    audio = audio.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        n_samples = int(len(audio) * sr / orig_sr)
        audio = scipy_resample(audio, n_samples).astype(np.float32)
    return audio, sr

_align_mod.load_audio = _fixed_load_audio

# Vervang de volledige build functie met batch versie
def _build_index_batched(file_paths, aligner, embedder, batch_size=64):
    import gc
    from tqdm import tqdm
    from audio_matcher.index import IndexEntry, build_phoneme_index
    from audio_matcher.alignment import _extract_audio_segment

    all_embeddings = []
    all_entries = []
    entry_counter = 0

    for path in tqdm(file_paths, desc="Building index"):
        try:
            phonemes = aligner.get_phonemes(str(path))
        except Exception as e:
            print(f"  Warning: {path.name}: {e}")
            continue

        if not phonemes:
            continue

        try:
            audio, sr = _fixed_load_audio(str(path))
        except Exception as e:
            print(f"  Warning load: {path.name}: {e}")
            continue

        print(f"  [{path.name}] {len(phonemes)} phonemes")

        # Verzamel segmenten
        segs, phs_valid = [], []
        for ph in phonemes:
            seg = _extract_audio_segment(audio, sr, ph.start, ph.end)
            if len(seg) >= sr * 0.05:
                segs.append(seg)
                phs_valid.append(ph)

        # Batch embed
        for i in range(0, len(segs), batch_size):
            batch_segs = segs[i:i+batch_size]
            batch_phs  = phs_valid[i:i+batch_size]

            # Pad naar gelijke lengte
            max_len = max(len(s) for s in batch_segs)
            padded = np.zeros((len(batch_segs), max_len), dtype=np.float32)
            for j, s in enumerate(batch_segs):
                padded[j, :len(s)] = s

            try:
                inputs = embedder.processor(
                    list(padded), sampling_rate=16000,
                    return_tensors='pt', padding=True
                )
                input_values = inputs.input_values.to(embedder.device)
                with torch.no_grad():
                    hidden = embedder.model(input_values).last_hidden_state
                    embs = hidden.mean(dim=1).cpu().numpy()
                norms = np.linalg.norm(embs, axis=1, keepdims=True)
                embs = embs / (norms + 1e-8)
            except Exception as e:
                print(f"  batch embed error: {e}")
                continue

            for emb, ph in zip(embs, batch_phs):
                all_embeddings.append(emb.reshape(1, -1))
                all_entries.append(IndexEntry(
                    phoneme=ph.phoneme,
                    start_time=ph.start,
                    end_time=ph.end,
                    word=ph.word,
                    audio_idx=entry_counter,
                    source_file=str(path),
                ))
                entry_counter += 1

        del audio, segs
        gc.collect()
        torch.cuda.empty_cache()

    all_embs = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 768), dtype='float32')
    print(f"Built index: {len(all_entries)} phonemes from {len(file_paths)} files")
    return build_phoneme_index(all_embs, all_entries)

pindex = _build_index_batched(files, aligner, pipeline, batch_size=64)
print(f'✅ entries: {len(pindex.entries)}')

Building index:   0%|          | 0/30 [00:00<?, ?it/s]

2026-06-03 11:16:33 - whisperx.asr - INFO - Detected language: en (0.81) in first 30s of audio
  [audio.wav] 9375 phonemes


Building index:   3%|▎         | 1/30 [00:38<18:49, 38.96s/it]

2026-06-03 11:17:12 - whisperx.asr - INFO - Detected language: en (0.82) in first 30s of audio
  [audio.wav] 9231 phonemes


Building index:   7%|▋         | 2/30 [01:15<17:29, 37.50s/it]

...

2026-06-03 11:35:03 - whisperx.asr - INFO - Detected language: en (0.80) in first 30s of audio
  [audio.wav] 10025 phonemes


Building index:  97%|█████████▋| 29/30 [19:09<00:40, 40.50s/it]

2026-06-03 11:35:43 - whisperx.asr - INFO - Detected language: en (0.80) in first 30s of audio
  [audio.wav] 7946 phonemes


Building index: 100%|██████████| 30/30 [19:42<00:00, 39.43s/it]


Built index: 173096 phonemes from 30 files
✅ entries: 173096


## 8. Pipeline uitvoeren + exporteren

In [ ]:
import joblib
save_path = str(DRIVE_ROOT / 'output/phoneme_index.joblib')
joblib.dump(pindex, save_path)
print(f'✅ Index opgeslagen: {save_path}')

✅ Index opgeslagen: /content/drive/MyDrive/projecten/Video_Analyzer_data/output/phoneme_index.joblib


In [9]:
import joblib
save_path = str(DRIVE_ROOT / 'output/phoneme_index.joblib')
# joblib.dump(pindex, save_path)

# Laden
pindex = joblib.load(save_path)
print(f"✅ Index succesvol geladen! Aantal entries: {len(pindex.entries)}")

✅ Index succesvol geladen! Aantal entries: 173096


In [10]:
# 1. Installeer torchcrepe (GPU pitch extraction, geen numba)
!pip install -q torchcrepe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 73.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyannote-metrics 4.1 requires numpy>=2.2.2, but you have numpy 2.0.2 which is incompatible.
whisperx 3.8.6 requires numpy>=2.1.0, but you have numpy 2.0.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you 

In [11]:
# 3. Gebruik
from audio_matcher.concatenative_synth import build_phoneme_library, run_concatenative_pipeline

In [12]:
import numpy as np
import torch
from scipy.signal import resample as scipy_resample
import audio_matcher.alignment as _align_mod

def _fixed_load_audio(file_path: str, sr: int = 16000):
    import soundfile as _sf
    audio, orig_sr = _sf.read(file_path, always_2d=True)
    audio = audio.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        n_samples = int(len(audio) * sr / orig_sr)
        audio = scipy_resample(audio, n_samples).astype(np.float32)
    return audio, sr

In [13]:
# Stap 1: bouw bibliotheek (eenmalig, ~5-10 min)
phoneme_lib = build_phoneme_library(
    episode_files=files,
    aligner=aligner,
    load_audio_fn=_fixed_load_audio,
    max_samples_per_type=10,  # max 10 samples per klank
)

Building phoneme library:   0%|          | 0/30 [00:00<?, ?it/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/145M [00:00<?, ?B/s]

...

2026-06-06 14:56:05 - whisperx.asr - INFO - Detected language: en (0.80) in first 30s of audio


Building phoneme library:  97%|█████████▋| 29/30 [14:50<00:30, 30.71s/it]

2026-06-06 14:56:35 - whisperx.asr - INFO - Detected language: en (0.80) in first 30s of audio


Building phoneme library: 100%|██████████| 30/30 [15:15<00:00, 30.53s/it]

✅ Bibliotheek: Voiced types: 74 (281 samples) | Unvoiced types: 34 (110 samples)


In [16]:
save_path = str(DRIVE_ROOT / 'output/phoneme_lib.joblib')
joblib.dump(phoneme_lib, save_path)
print(f'✅ Index opgeslagen: {save_path}')

✅ Index opgeslagen: /content/drive/MyDrive/projecten/Video_Analyzer_data/output/phoneme_lib.joblib


In [17]:
save_path = str(DRIVE_ROOT / 'output/phoneme_lib.joblib')

# Laden
phoneme_lib1 = joblib.load(save_path)
print(f"✅ Index succesvol geladen! Aantal entries: {len(pindex.entries)}")

✅ Index succesvol geladen! Aantal entries: 173096


In [18]:
# Stap 2: syntheseer
result = run_concatenative_pipeline(
    song_path=SONG_PATH,
    phoneme_library=phoneme_lib1,
    aligner=aligner,
    load_audio_fn=_fixed_load_audio,
    crossfade_ms=20,
    pitch_shift_enabled=True,
)

🎵 Stap 1: song laden + normaliseren...
   Duur: 231.4s @ 16000Hz
🎵 Stap 2: phoneme + pitch extractie van song...
2026-06-06 14:59:36 - whisperx.asr - INFO - Detected language: en (0.84) in first 30s of audio
   1981 phonemes
   Voiced frames: 1978/1981 (99%)
🎵 Stap 3: synthese per phoneme...


   Synthese: 100%|██████████| 1981/1981 [00:00<00:00, 2481.50it/s]


   Voiced hits: 1978, Unvoiced hits: 3, Misses: 0
✅ Output: 231.4s, max dBFS: -0.4


In [19]:
export_audio(result, str(DRIVE_ROOT / 'output/concatenative_v1.wav'))

In [ ]:
OUTPUT_PATH = str(DRIVE_ROOT / 'output/aligned_output_colab1.wav')

final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
print(f'Output lengte: {len(final_audio)} ms')

export_audio(final_audio, OUTPUT_PATH)
print(f'✅ Opgeslagen: {OUTPUT_PATH}')

2026-06-03 10:34:06 - whisperx.asr - INFO - Detected language: en (0.84) in first 30s of audio
song phonemes: 1981, matched: 1312
output_ms: 448699
Output lengte: 448699 ms
✅ Opgeslagen: /content/drive/MyDrive/projecten/Video_Analyzer_data/output/aligned_output_colab1.wav
